# Deep Convolutional Q-Learning for Pac-Man

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [1]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
"apt-get" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


INFO: pip is looking at multiple versions of gymnasium[box2d] to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 14.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.8 MB 14.4 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.8 MB 15.0 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 15.3 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 14.5 MB/s  0:00:00


  error: subprocess-exited-with-error
  
  × Building wheel for box2d-py (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [29 lines of output]
      Using setuptools (version 84.2.1).
      C:\Users\gloria.segurini\AppData\Local\Temp\pip-build-env-kbn4zsvr\overlay\Lib\site-packages\setuptools\_distutils\dist.py:318: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\gloria.segurini\AppData\Local\Temp\pip-build-env-kbn4zsvr\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: zlib/libpng License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for deta

### Importing the libraries

In [3]:
pip install torch

  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
    --------------------------------------- 1.6/124.1 MB 8.7 MB/s eta 0:00:15
   - -------------------------------------- 4.7/124.1 MB 12.2 MB/s eta 0:00:10
   -- ------------------------------------- 8.1/124.1 MB 13.4 MB/s eta 0:00:09
   --- ------------------------------------ 11.3/124.1 MB 14.0 MB/s eta 0:00:09
   ---- ----------------------------------- 14.4/124.1 MB 14.3 MB/s eta 0:00:08
   ----- ---------------------------------- 17.8/124.1 MB 14.5 MB/s eta 0:00:08
   ------ --------------------------------- 21.2/124.1 MB 14.7 MB/s eta 0:00:08
   ------- -------------------------------- 24.1/124.1 MB 14.7 MB/s eta 0:00:07
   -------- -------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [5]:
class Network(nn.Module):

  def __init__(self, action_size, seed=42):
    # Eyes of the CNN

    # Initializes the base nn.Module class so that all PyTorch layers
    # are properly registered.
    super(Network, self).__init__()

    # Sets the PyTorch random seed so results are reproducible.
    self.seed = torch.manual_seed(seed)

    # 1st convolution: takes RGB images (3 channels) and extracts
    # 32 feature maps.
    # An 8x8 kernel with a stride of 4 significantly reduces
    # the image width and height.
    self.conv1 = nn.Conv2d(3, 32, kernel_size=8, stride=4)

    # Normalizes the 32 feature maps produced by conv1
    # to make training more stable.
    # Batch Normalization keeps the activations of each channel
    # in a more controlled distribution.
    self.bn1 = nn.BatchNorm2d(32)

    # 2nd convolution: combines the previous 32 feature maps
    # into 64 more abstract features.
    # Gradually increases the depth (# of channels) while
    # decreasing the spatial dimensions.
    self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)

    # Batch Normalization applied to the 64 feature maps from conv2.
    self.bn2 = nn.BatchNorm2d(64)

    # 3rd convolution: keeps 64 channels and looks for more local
    # details using a 3x3 kernel.
    self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

    # Batch Normalization applied to the 64 feature maps from conv3.
    self.bn3 = nn.BatchNorm2d(64)

    # 4th convolution: increases the number of feature maps
    # from 64 to 128, creating richer visual representations.
    self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1)

    # Batch Normalization applied to the 128 feature maps from conv4.
    self.bn4 = nn.BatchNorm2d(128)

    # Brain of the CNN

    # After the convolutional layers, the output consists of
    # 128 feature maps of size 10x10.
    # These 12,800 values are flattened and connected to 512 neurons.
    # This input size depends on the resolution of the original image.
    self.fc1 = nn.Linear(10 * 10 * 128, 512)

    # Reduces the representation from 512 to 256 neurons.
    self.fc2 = nn.Linear(512, 256)

    # Produces one Q-value for each possible action.
    # These are not probabilities, but estimates of expected return.
    self.fc3 = nn.Linear(256, action_size)

  def forward(self, state):
    x = F.relu(self.bn1(self.conv1(state)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))

    # Flattens the convolutional output while keeping
    # the batch dimension unchanged.
    x = x.view(x.size(0), -1)

    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))

    return self.fc3(x)